# Build the Mammo-Bench control set and push it to Kaggle

**Run this in Colab.** About five minutes, and it saves downloading and re-uploading several
gigabytes — the data goes Drive → Kaggle directly and never touches your laptop.

The v13 preprocessing control needs the **2,821 Mammo-Bench test images** plus `metadata.csv`,
not the whole 19,731-image dataset. This notebook selects the test split from the frozen split
file, zips it, and uploads it as a private Kaggle dataset.

## Before you start

Kaggle's API token changed format. It is now a single string beginning `KGAT_`, not the old
`kaggle.json` with a username and key. Get one at **kaggle.com → your avatar → Settings →
API → Create New Token**. This notebook accepts either format.

**Treat that token like a password.** It grants full access to your account. Do not paste it
into a cell, a screenshot, or anything you share — cell 1 uses a hidden prompt so it never
gets written into the notebook file. If a token has been exposed, expire it on the same
settings page and generate a new one.

## Why the control matters

Without it, if RSNA comes back low you cannot tell whether the model fails to generalise or
your DICOM pipeline simply differs from Mammo-Bench's. With it, v13 reports the two
separately. That is the difference between a result and an unexplained number.

## 1. Authenticate first

Deliberately before anything slow — a bad token should fail in seconds, not after a
five-minute zip.

In [ ]:
import os, subprocess, json, getpass
subprocess.run('pip install -q --upgrade kagglehub requests', shell=True)
import requests

# Your Kaggle username. It is the first part of your dataset URLs, and it appeared as
# /kaggle/input/datasets/<username>/... in the v13 run.
KAGGLE_USERNAME = 'pasindupahasara'
DATASET_SLUG    = 'mammobench-test-control'

KAGGLE_DIR = os.path.expanduser('~/.kaggle')
os.makedirs(KAGGLE_DIR, exist_ok=True)
token = None

# 1. a new-style token already on this machine or in Drive
for p in [os.path.join(KAGGLE_DIR, 'access_token'),
          '/content/drive/MyDrive/access_token']:
    if os.path.exists(p):
        token = open(p).read().strip()
        print('found an access_token at', p); break

# 2. an old-style kaggle.json
if not token:
    for p in [os.path.join(KAGGLE_DIR, 'kaggle.json'),
              '/content/drive/MyDrive/kaggle.json']:
        if os.path.exists(p):
            j = json.load(open(p))
            os.environ['KAGGLE_USERNAME'] = j['username']
            os.environ['KAGGLE_KEY'] = j['key']
            KAGGLE_USERNAME = j['username']
            print('using the old-style kaggle.json at', p, 'as', KAGGLE_USERNAME)
            token = 'legacy'; break

# 3. ask, hidden - so it never lands in the saved notebook
if not token:
    token = getpass.getpass('Paste your Kaggle token (starts KGAT_), input is hidden: ').strip()

if token and token != 'legacy':
    os.environ['KAGGLE_API_TOKEN'] = token
    with open(os.path.join(KAGGLE_DIR, 'access_token'), 'w') as f:
        f.write(token)
    os.chmod(os.path.join(KAGGLE_DIR, 'access_token'), 0o600)

# ---- verify before doing anything expensive ----
ok = False
if token and token != 'legacy':
    r = requests.get('https://www.kaggle.com/api/v1/datasets/list',
                     headers={'Authorization': f'Bearer {token}'}, timeout=30)
    ok = (r.status_code == 200)
    print('token check:', r.status_code, 'OK' if ok else r.text[:200])
else:
    r = subprocess.run('kaggle datasets list -m --page-size 1', shell=True,
                       capture_output=True, text=True)
    ok = (r.returncode == 0)
    print('kaggle.json check:', 'OK' if ok else (r.stderr or r.stdout)[:200])

assert ok, ('Authentication failed. Generate a fresh token at kaggle.com/settings/api. '
            'If you just expired one, the old string will no longer work.')
print(f'\nauthenticated. target dataset: {KAGGLE_USERNAME}/{DATASET_SLUG}')

Paste your Kaggle token (starts KGAT_), input is hidden: ··········
token check: 200 OK

authenticated. target dataset: pasindupahasara/mammobench-test-control


## 2. Mount Drive and locate the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, shutil
import pandas as pd

DRIVE_ROOT  = '/content/drive/MyDrive/momobench-dataset'
EXTRACT_DIR = '/content/dataset_original'
IMG_ROOT    = os.path.join(EXTRACT_DIR, 'dataset')
SPLIT_PATH  = os.path.join(DRIVE_ROOT, 'sprint4_audit', 'frozen_split.json')

if not os.path.isdir(IMG_ROOT):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print('extracting dataset.zip (a few minutes)...')
    os.system(f'unzip -q "{DRIVE_ROOT}/dataset.zip" -d "{EXTRACT_DIR}"')

assert os.path.isdir(IMG_ROOT), f'{IMG_ROOT} missing - check DRIVE_ROOT'
assert os.path.exists(SPLIT_PATH), f'{SPLIT_PATH} missing'
print('images ->', IMG_ROOT)
print('split  ->', SPLIT_PATH)

Mounted at /content/drive
extracting dataset.zip (a few minutes)...
images -> /content/dataset_original/dataset
split  -> /content/drive/MyDrive/momobench-dataset/sprint4_audit/frozen_split.json


## 3. Select the test split

In [ ]:
mb = pd.read_csv(os.path.join(IMG_ROOT, 'metadata.csv'))
mb['classification'] = mb['classification'].replace({'Suspicious Malignant': 'Malignant'})
mb = mb[mb['source_dataset'] != 'cdd-cesm'].reset_index(drop=True)

test_ids = set(json.load(open(SPLIT_PATH))['test'])
sel = mb[mb['source_subjectID'].isin(test_ids)].reset_index(drop=True)
sel['full'] = sel['preprocessed_image_path'].apply(lambda p: os.path.join(IMG_ROOT, p))
sel = sel[sel['full'].apply(os.path.exists)].reset_index(drop=True)

print(f'test images : {len(sel)}   (expected 2,821)')
print(f'patients    : {sel.source_subjectID.nunique()}   (expected 831)')
print('\nclass balance:'); print(sel.classification.value_counts().to_string())
print('\nby source:');     print(sel.source_dataset.value_counts().to_string())

probe = sum(os.path.getsize(p) for p in sel['full'].head(300))
print(f'\nestimated upload size: {probe/300*len(sel)/1e6:.0f} MB')

test images : 2821   (expected 2,821)
patients    : 831   (expected 831)

class balance:
classification
Malignant    1256
Benign        859
Normal        706

by source:
source_dataset
ddsm        1528
cmmd         826
kau-bcmd     330
dmid          76
inbreast      61

estimated upload size: 1404 MB


## 4. Build the upload folder

The **full** `metadata.csv` goes in, not the filtered one — v13 filters it again itself, and
keeping it whole means it is byte-identical to the file every other notebook reads.

In [ ]:
STAGE = '/content/kaggle_control'
shutil.rmtree(STAGE, ignore_errors=True)
os.makedirs(STAGE, exist_ok=True)

zip_path = os.path.join(STAGE, 'mammobench_test.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as z:   # JPGs are already compressed
    z.write(os.path.join(IMG_ROOT, 'metadata.csv'), 'metadata.csv')
    for i, (_, r) in enumerate(sel.iterrows()):
        z.write(r['full'], r['preprocessed_image_path'])
        if (i + 1) % 500 == 0: print(f'  {i+1}/{len(sel)}')

# also loose, so metadata.csv is findable even if the zip is not expanded
shutil.copy(os.path.join(IMG_ROOT, 'metadata.csv'), os.path.join(STAGE, 'metadata.csv'))

print(f'\nstaged {sorted(os.listdir(STAGE))}')
print(f'zip: {os.path.getsize(zip_path)/1e6:.0f} MB')

  500/2821
  1000/2821
  1500/2821
  2000/2821
  2500/2821

staged ['mammobench_test.zip', 'metadata.csv']
zip: 684 MB


## 5. Upload

In [ ]:
import kagglehub

handle = f'{KAGGLE_USERNAME}/{DATASET_SLUG}'
print(f'uploading to {handle} - this is the slow part\n')

try:
    kagglehub.dataset_upload(handle, STAGE,
                             version_notes='Mammo-Bench test split, v13 preprocessing control')
    print('\ndone via kagglehub')
except Exception as e:
    print('kagglehub upload failed:', repr(e))
    print('falling back to the kaggle CLI...\n')
    subprocess.run('pip install -q --upgrade kaggle', shell=True)
    json.dump({'title': 'Mammo-Bench test control', 'id': handle,
               'licenses': [{'name': 'CC0-1.0'}]},
              open(os.path.join(STAGE, 'dataset-metadata.json'), 'w'), indent=2)
    r = subprocess.run(f'kaggle datasets create -p {STAGE} --dir-mode zip',
                       shell=True, capture_output=True, text=True)
    out = (r.stdout or '') + (r.stderr or '')
    print(out)
    if 'already exists' in out:
        r = subprocess.run(f'kaggle datasets version -p {STAGE} -m update --dir-mode zip',
                           shell=True, capture_output=True, text=True)
        print(r.stdout or '', r.stderr or '')

print(f'\nkaggle.com/datasets/{handle}')

uploading to pasindupahasara/mammobench-test-control - this is the slow part

Uploading Dataset https://kaggle.com/datasets/pasindupahasara/mammobench-test-control ...
Starting upload for file /content/kaggle_control/metadata.csv


Uploading: 100%|██████████| 4.18M/4.18M [00:00<00:00, 14.8MB/s]

Upload successful: /content/kaggle_control/metadata.csv (4MB)
Starting upload for file /content/kaggle_control/mammobench_test.zip



Uploading: 100%|██████████| 684M/684M [00:06<00:00, 102MB/s]

Upload successful: /content/kaggle_control/mammobench_test.zip (652MB)


Your dataset has been created.
Files are being processed...
See at: https://kaggle.com/datasets/pasindupahasara/mammobench-test-control

done via kagglehub

kaggle.com/datasets/pasindupahasara/mammobench-test-control


## 6. Then, in the v13 notebook on Kaggle

1. **+ Add Input → Datasets → Your Datasets →** `mammobench-test-control`
2. Re-run section 2. It should now print `control set -> .../metadata.csv` instead of the
   "preprocessing control DISABLED" warning.

You do not need to set `KAGGLE_MAMMOBENCH_DATASET` — v13 searches for `metadata.csv` by name
wherever Kaggle mounts it, and falls back to matching image basenames if the upload flattened
the folder tree.

**What it buys.** Section 6a re-runs the v12 ensemble on these images through the same
pipeline RSNA goes through, and reports the gap against the known internal 0.7643. Any drop
beyond that gap on RSNA is population shift rather than preprocessing — which is what makes
the external number reportable whichever way it lands.

**Housekeeping.** If you pasted a token that has since been exposed anywhere, expire it at
kaggle.com/settings/api and generate a new one. Uploading does not require keeping the old
one alive.